In [0]:
# ==============================================================================
# MILESTONE 5 - TASK 5.1: DAILY SALES METRICS & WINDOW FUNCTIONS
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 80)
print("INITIALIZING TASK 5.1: GOLD DAILY SALES METRICS")
print("=" * 80)

# Load Silver Datasets
silver_orders = spark.table("globalmart.silver.silver_orders")
silver_order_items = spark.table("globalmart.silver.silver_order_items")

print("✅ Silver tables loaded successfully.")

In [0]:
# ==============================================================================
# STEP 1: BASE DAILY REVENUE & ORDER COUNT AGGREGATION
# ==============================================================================

# Filter delivered orders and extract purchase date
daily_base_df = (
    silver_orders.filter(F.col("order_status") == "delivered")
    .withColumn("order_date", F.to_date(F.col("order_purchase_timestamp")))
    .join(silver_order_items, on="order_id", how="inner")
    .groupBy("order_date")
    .agg(
        F.round(F.sum("total_item_value"), 2).alias("daily_revenue"),
        F.countDistinct("order_id").alias("daily_order_count")
    )
    .filter(F.col("order_date").isNotNull())
)

print(f"• Total distinct sales days processed: {daily_base_df.count():,}")

In [0]:
# ==============================================================================
# STEP 2: APPLY WINDOW FUNCTIONS (CUMULATIVE, MOVING AVG, DOD, MONTHLY RANK)
# ==============================================================================

# 1. Window for Cumulative Running Total (Overall timeline)
w_cumulative = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 2. Window for 7-Day Moving Average
w_7d_ma = Window.orderBy("order_date").rowsBetween(-6, Window.currentRow)

# 3. Window for 30-Day Moving Average
w_30d_ma = Window.orderBy("order_date").rowsBetween(-29, Window.currentRow)

# 4. Window for Day-over-Day (DoD) Lag
w_lag = Window.orderBy("order_date")

# 5. Window for Monthly Revenue Rank
w_monthly_rank = Window.partitionBy(F.trunc("order_date", "month")).orderBy(F.col("daily_revenue").desc())

# Calculate Metrics
gold_daily_sales_df = (
    daily_base_df
    # Cumulative Revenue
    .withColumn("cumulative_revenue", F.round(F.sum("daily_revenue").over(w_cumulative), 2))
    
    # 7-Day & 30-Day Moving Averages
    .withColumn("moving_avg_7d", F.round(F.avg("daily_revenue").over(w_7d_ma), 2))
    .withColumn("moving_avg_30d", F.round(F.avg("daily_revenue").over(w_30d_ma), 2))
    
    # Day-over-Day Changes
    .withColumn("prev_day_revenue", F.lag("daily_revenue", 1).over(w_lag))
    .withColumn("dod_absolute_change", F.round(F.col("daily_revenue") - F.coalesce(F.col("prev_day_revenue"), F.col("daily_revenue")), 2))
    .withColumn(
        "dod_pct_change", 
        F.round(
            F.when(F.col("prev_day_revenue").isNull() | (F.col("prev_day_revenue") == 0), 0.0)
             .otherwise(((F.col("daily_revenue") - F.col("prev_day_revenue")) / F.col("prev_day_revenue")) * 100), 
            2
        )
    )
    
    # Monthly Revenue Rank
    .withColumn("monthly_revenue_rank", F.dense_rank().over(w_monthly_rank))
    .withColumn("_gold_processed_at", F.current_timestamp())
    .drop("prev_day_revenue")
    .orderBy("order_date")
)

print("✅ Gold window metrics computed successfully.")

In [0]:
# ==============================================================================
# STEP 3: PERSIST TO GOLD DELTA TABLE
# ==============================================================================

gold_table_name = "globalmart.gold.gold_daily_sales_metrics"

(
    gold_daily_sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_name)
)

print(f"✅ Successfully created/overwritten Gold Table: {gold_table_name}")

In [0]:
# ==============================================================================
# TASK 5.1 DELIVERABLE: SINGLE MONTH SAMPLE OUTPUT (NOVEMBER 2017)
# ==============================================================================

month_sample_df = (
    spark.table("globalmart.gold.gold_daily_sales_metrics")
    .filter((F.col("order_date") >= "2017-11-01") & (F.col("order_date") <= "2017-11-30"))
    .select(
        "order_date",
        "daily_revenue",
        "daily_order_count",
        "cumulative_revenue",
        "moving_avg_7d",
        "moving_avg_30d",
        "dod_absolute_change",
        "dod_pct_change",
        "monthly_revenue_rank"
    )
    .orderBy("order_date")
)

print("=" * 80)
print("TASK 5.1 DELIVERABLE: DAILY SALES METRICS FOR NOVEMBER 2017")
print("=" * 80)
display(month_sample_df)

In [0]:
# ==============================================================================
# MILESTONE 5 - TASK 5.2: CUSTOMER RFM SEGMENTATION
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 80)
print("INITIALIZING TASK 5.2: GOLD CUSTOMER RFM SEGMENTATION")
print("=" * 80)

# Load Silver Tables
silver_orders = spark.table("globalmart.silver.silver_orders")
silver_order_items = spark.table("globalmart.silver.silver_order_items")
silver_customers = spark.table("globalmart.silver.silver_customers")

# 1. Determine Dataset Reference Date (Last order date in dataset)
max_date_row = silver_orders.filter(F.col("order_status") == "delivered") \
    .select(F.max(F.to_date("order_purchase_timestamp")).alias("max_date")).collect()

ref_date = max_date_row[0]["max_date"]
print(f"📅 Dataset Reference Date for Recency Calculations: {ref_date}")

# 2. Compute Raw R, F, M per Customer Unique ID
# Note: Using customer_unique_id to group repeat purchases across different orders
cust_rfm_raw = (
    silver_orders.filter(F.col("order_status") == "delivered")
    .join(silver_customers, on="customer_id", how="inner")
    .join(silver_order_items, on="order_id", how="inner")
    .groupBy("customer_unique_id")
    .agg(
        F.datediff(F.lit(ref_date), F.max(F.to_date("order_purchase_timestamp"))).alias("recency_days"),
        F.countDistinct("order_id").alias("frequency_orders"),
        F.round(F.sum("total_item_value"), 2).alias("monetary_spend")
    )
)

print(f"• Unique Customers Processed: {cust_rfm_raw.count():,}")

In [0]:
# ==============================================================================
# STEP 2: SCORE R, F, M AND MAP TO NAMED SEGMENTS
# ==============================================================================

# Recency Window: Ascending recency_days (fewer days = higher score)
w_recency = Window.orderBy(F.col("recency_days").desc())
# Monetary Window: Ascending monetary_spend (higher spend = higher score)
w_monetary = Window.orderBy(F.col("monetary_spend").asc())

rfm_scored_df = (
    cust_rfm_raw
    # Recency Score 1 to 5 (NTILE over descending days)
    .withColumn("r_score", F.ntile(5).over(w_recency))
    
    # Monetary Score 1 to 5 (NTILE over ascending spend)
    .withColumn("m_score", F.ntile(5).over(w_monetary))
    
    # Frequency Score 1 to 5 (Explicit mapping due to retail order frequency distribution)
    .withColumn(
        "f_score",
        F.when(F.col("frequency_orders") >= 3, 5)
         .when(F.col("frequency_orders") == 2, 3)
         .otherwise(1)
    )
    # Composite RFM Score String (e.g., '555')
    .withColumn("rfm_combined_score", F.concat("r_score", "f_score", "m_score"))
)

# Define Segment Mapping Logic
rfm_segmented_df = (
    rfm_scored_df
    .withColumn(
        "rfm_segment",
        F.when((F.col("r_score") >= 4) & (F.col("f_score") >= 3) & (F.col("m_score") >= 4), "Champions")
         .when((F.col("r_score") >= 3) & (F.col("f_score") >= 3), "Loyal Customers")
         .when((F.col("r_score") >= 4) & (F.col("f_score") <= 2), "Recent / New Buyers")
         .when((F.col("r_score") <= 2) & (F.col("m_score") >= 4), "At Risk / High Value")
         .when((F.col("r_score") <= 2) & (F.col("f_score") <= 2) & (F.col("m_score") <= 2), "Lost / Inactive")
         .otherwise("Promising / Hibernating")
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

print("✅ RFM Scoring and Segment Mapping complete.")

In [0]:
# ==============================================================================
# STEP 3: PERSIST TO GOLD DELTA TABLE
# ==============================================================================

gold_table_name = "globalmart.gold.gold_customer_rfm_segmentation"

(
    rfm_segmented_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_name)
)

print(f"✅ Successfully created Gold Table: {gold_table_name}")

In [0]:
# ==============================================================================
# TASK 5.2 DELIVERABLE: SEGMENT DISTRIBUTION REPORT
# ==============================================================================

total_customers = spark.table("globalmart.gold.gold_customer_rfm_segmentation").count()

segment_dist_df = (
    spark.table("globalmart.gold.gold_customer_rfm_segmentation")
    .groupBy("rfm_segment")
    .agg(
        F.count("customer_unique_id").alias("customer_count"),
        F.round((F.count("customer_unique_id") / F.lit(total_customers)) * 100, 2).alias("pct_of_total"),
        F.round(F.avg("recency_days"), 1).alias("avg_recency_days"),
        F.round(F.avg("frequency_orders"), 2).alias("avg_frequency_orders"),
        F.round(F.avg("monetary_spend"), 2).alias("avg_monetary_spend"),
        F.round(F.sum("monetary_spend"), 2).alias("total_segment_revenue")
    )
    .orderBy(F.col("total_segment_revenue").desc())
)

print("=" * 80)
print("TASK 5.2 DELIVERABLE: CUSTOMER RFM SEGMENT DISTRIBUTION")
print("=" * 80)
display(segment_dist_df)

In [0]:
%sql
WITH monthly_category_revenue AS (
    SELECT 
        c.category_name,
        DATE_TRUNC('month', o.order_purchase_timestamp) AS revenue_month,
        SUM(oi.price) AS monthly_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN categories c ON p.category_id = c.category_id
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY 1, 2
),
mom_growth AS (
    SELECT 
        category_name,
        revenue_month,
        monthly_revenue,
        LAG(monthly_revenue) OVER (PARTITION BY category_name ORDER BY revenue_month) AS prev_month_revenue,
        (monthly_revenue - LAG(monthly_revenue) OVER (PARTITION BY category_name ORDER BY revenue_month)) 
            / NULLIF(LAG(monthly_revenue) OVER (PARTITION BY category_name ORDER BY revenue_month), 0) * 100 AS mom_growth_pct
    FROM monthly_category_revenue
),
growth_islands AS (
    SELECT 
        category_name,
        revenue_month,
        monthly_revenue,
        prev_month_revenue,
        mom_growth_pct,
        ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY revenue_month) - 
        ROW_NUMBER() OVER (PARTITION BY category_name, CASE WHEN mom_growth_pct > 0 THEN 1 ELSE 0 END ORDER BY revenue_month) AS streak_id
    FROM mom_growth
    WHERE mom_growth_pct > 0
)
SELECT 
    category_name AS category,
    DATE_FORMAT(MIN(revenue_month), 'yyyy-MM') AS streak_start_month,
    DATE_FORMAT(MAX(revenue_month), 'yyyy-MM') AS streak_end_month,
    COUNT(*) AS streak_length_months,
    ROUND(
        ((MAX(monthly_revenue) - MIN(prev_month_revenue)) / NULLIF(MIN(prev_month_revenue), 0) * 100), 
        2
    ) AS total_growth_pct
FROM growth_islands
GROUP BY category_name, streak_id
HAVING COUNT(*) >= 3
ORDER BY streak_length_months DESC, total_growth_pct DESC;

In [0]:
%sql
SHOW SCHEMAS;

In [0]:
%sql
SHOW TABLES IN globalmart.silver;

In [0]:
# ==============================================================================
# TASK 5.3: CATEGORY GROWTH STREAKS (GLOBALMART LAKEHOUSE)
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load available Silver tables
silver_orders = spark.table("globalmart.silver.silver_orders")
silver_order_items = spark.table("globalmart.silver.silver_order_items")

# 1. Base Aggregation: Monthly Revenue per Category for Delivered Orders
monthly_cat_revenue = (
    silver_orders.filter(F.col("order_status") == "delivered")
    .join(silver_order_items, "order_id", "inner")
    .withColumn("revenue_month", F.trunc(F.col("order_purchase_timestamp"), "month"))
    .groupBy("product_category_name", "revenue_month")
    .agg(F.round(F.sum("total_item_value"), 2).alias("monthly_revenue"))
    .filter(F.col("product_category_name").isNotNull())
)

# 2. Compute MoM Growth %
w_cat_month = Window.partitionBy("product_category_name").orderBy("revenue_month")

mom_df = (
    monthly_cat_revenue
    .withColumn("prev_month_revenue", F.lag("monthly_revenue", 1).over(w_cat_month))
    .withColumn(
        "mom_growth_pct",
        F.when(F.col("prev_month_revenue").isNull() | (F.col("prev_month_revenue") == 0), 0.0)
         .otherwise(((F.col("monthly_revenue") - F.col("prev_month_revenue")) / F.col("prev_month_revenue")) * 100)
    )
    .filter(F.col("mom_growth_pct") > 0)
)

# 3. Gaps & Islands Logic to Identify Consecutive Growth Streaks
w_cat_order = Window.partitionBy("product_category_name").orderBy("revenue_month")

growth_islands = mom_df.withColumn(
    "streak_id",
    F.row_number().over(w_cat_order) - F.row_number().over(w_cat_order)
)

# 4. Filter Streaks >= 3 Consecutive Months
streaks_summary = (
    growth_islands
    .groupBy("product_category_name", "streak_id")
    .agg(
        F.date_format(F.min("revenue_month"), "yyyy-MM").alias("streak_start_month"),
        F.date_format(F.max("revenue_month"), "yyyy-MM").alias("streak_end_month"),
        F.count("revenue_month").alias("streak_length_months"),
        F.first("prev_month_revenue").alias("start_base_revenue"),
        F.last("monthly_revenue").alias("end_revenue")
    )
    .filter(F.col("streak_length_months") >= 3)
    .withColumn(
        "total_growth_pct",
        F.round(((F.col("end_revenue") - F.col("start_base_revenue")) / F.col("start_base_revenue")) * 100, 2)
    )
    .select(
        F.col("product_category_name").alias("category"),
        "streak_start_month",
        "streak_end_month",
        "streak_length_months",
        "total_growth_pct"
    )
    .orderBy(F.col("streak_length_months").desc(), F.col("total_growth_pct").desc())
)

print("=" * 80)
print("TASK 5.3 DELIVERABLE: CATEGORY GROWTH STREAKS (>= 3 CONSECUTIVE MONTHS)")
print("=" * 80)
display(streaks_summary)

In [0]:
%sql
WITH monthly_category_revenue AS (
    SELECT 
        oi.product_category_name AS category,
        DATE_TRUNC('month', o.order_purchase_timestamp) AS revenue_month,
        SUM(oi.total_item_value) AS monthly_revenue
    FROM globalmart.silver.silver_orders o
    JOIN globalmart.silver.silver_order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered' 
      AND oi.product_category_name IS NOT NULL
    GROUP BY oi.product_category_name, DATE_TRUNC('month', o.order_purchase_timestamp)
),
mom_growth AS (
    SELECT 
        category,
        revenue_month,
        monthly_revenue,
        LAG(monthly_revenue) OVER (PARTITION BY category ORDER BY revenue_month) AS prev_month_revenue,
        (monthly_revenue - LAG(monthly_revenue) OVER (PARTITION BY category ORDER BY revenue_month)) 
            / NULLIF(LAG(monthly_revenue) OVER (PARTITION BY category ORDER BY revenue_month), 0) * 100 AS mom_growth_pct
    FROM monthly_category_revenue
),
growth_islands AS (
    SELECT 
        category,
        revenue_month,
        monthly_revenue,
        prev_month_revenue,
        mom_growth_pct,
        ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue_month) - 
        ROW_NUMBER() OVER (PARTITION BY category, CASE WHEN mom_growth_pct > 0 THEN 1 ELSE 0 END ORDER BY revenue_month) AS streak_group
    FROM mom_growth
    WHERE mom_growth_pct > 0
)
SELECT 
    category,
    DATE_FORMAT(MIN(revenue_month), 'yyyy-MM') AS streak_start_month,
    DATE_FORMAT(MAX(revenue_month), 'yyyy-MM') AS streak_end_month,
    COUNT(*) AS streak_length_months,
    ROUND(
        ((MAX(monthly_revenue) - MIN(prev_month_revenue)) / NULLIF(MIN(prev_month_revenue), 0) * 100), 
        2
    ) AS total_growth_pct
FROM growth_islands
GROUP BY category, streak_group
HAVING COUNT(*) >= 3
ORDER BY streak_length_months DESC, total_growth_pct DESC;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS globalmart.gold.gold_customer_summary (
    customer_unique_id STRING,
    total_orders BIGINT,
    total_spend DOUBLE,
    avg_order_value DOUBLE,
    first_order_date TIMESTAMP,
    last_order_date TIMESTAMP,
    is_active BOOLEAN,
    last_updated_at TIMESTAMP
)
USING DELTA;

In [0]:
# ==============================================================================
# TASK 5.4: GOLD LAYER CUSTOMER SUMMARY MERGE PIPELINE (CORRECTED API SYNTAX)
# ==============================================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

def merge_customer_summary(cutoff_date_str="2018-01-01"):
    """
    Computes customer lifetime metrics and performs a Delta Lake MERGE to:
      1. INSERT new customers.
      2. UPDATE existing customers whose metrics/status have changed.
      3. SOFT-DELETE (set is_active = False) for customers without orders since cutoff_date_str.
    """
    target_table_name = "globalmart.gold.gold_customer_summary"
    cutoff_timestamp = F.to_timestamp(F.lit(cutoff_date_str))

    # 1. Source Data Aggregation
    orders = spark.table("globalmart.silver.silver_orders")
    order_items = spark.table("globalmart.silver.silver_order_items")

    order_spend = (
        order_items
        .groupBy("order_id")
        .agg(F.sum("total_item_value").alias("order_total_spend"))
    )

    customer_metrics = (
        orders.filter(F.col("order_status") == "delivered")
        .join(order_spend, "order_id", "left")
        .groupBy("customer_id")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.round(F.coalesce(F.sum("order_total_spend"), F.lit(0.0)), 2).alias("total_spend"),
            F.min("order_purchase_timestamp").alias("first_order_date"),
            F.max("order_purchase_timestamp").alias("last_order_date")
        )
        .withColumn("avg_order_value", F.round(F.col("total_spend") / F.col("total_orders"), 2))
        .withColumn("is_active", F.when(F.col("last_order_date") >= cutoff_timestamp, True).otherwise(False))
        .withColumn("last_updated_at", F.current_timestamp())
        .withColumnRenamed("customer_id", "customer_unique_id")
    )

    # 2. Perform Delta Lake MERGE
    target_delta = DeltaTable.forName(spark, target_table_name)

    (
        target_delta.alias("target")
        .merge(
            source=customer_metrics.alias("source"),
            condition="target.customer_unique_id = source.customer_unique_id"
        )
        # Match & Update uses set={...}
        .whenMatchedUpdate(
            condition="""
                target.total_orders != source.total_orders OR
                target.total_spend != source.total_spend OR
                target.is_active != source.is_active OR
                target.last_order_date != source.last_order_date
            """,
            set={
                "total_orders": "source.total_orders",
                "total_spend": "source.total_spend",
                "avg_order_value": "source.avg_order_value",
                "first_order_date": "source.first_order_date",
                "last_order_date": "source.last_order_date",
                "is_active": "source.is_active",
                "last_updated_at": "source.last_updated_at"
            }
        )
        # Not Matched & Insert uses values={...}
        .whenNotMatchedInsert(
            values={
                "customer_unique_id": "source.customer_unique_id",
                "total_orders": "source.total_orders",
                "total_spend": "source.total_spend",
                "avg_order_value": "source.avg_order_value",
                "first_order_date": "source.first_order_date",
                "last_order_date": "source.last_order_date",
                "is_active": "source.is_active",
                "last_updated_at": "source.last_updated_at"
            }
        )
        .execute()
    )

    # 3. Retrieve Operation Metrics & Soft-Delete Counts
    history_df = target_delta.history(1).select("operationMetrics")
    metrics = history_df.collect()[0]["operationMetrics"]

    num_inserted = int(metrics.get("numTargetRowsInserted", 0))
    num_updated = int(metrics.get("numTargetRowsUpdated", 0))
    num_soft_deleted = spark.table(target_table_name).filter(F.col("is_active") == False).count()

    print("=" * 80)
    print("TASK 5.4 DELIVERABLE: CUSTOMER SUMMARY MERGE METRICS")
    print("=" * 80)
    print(f"Active Status Cutoff Date : {cutoff_date_str}")
    print(f"Rows Inserted             : {num_inserted:,}")
    print(f"Rows Updated              : {num_updated:,}")
    print(f"Rows Soft-Deleted         : {num_soft_deleted:,}")
    print("=" * 80)

# Run the pipeline
merge_customer_summary(cutoff_date_str="2018-01-01")

In [0]:
%sql
SELECT 
    is_active,
    COUNT(*) AS total_customers,
    ROUND(AVG(total_orders), 2) AS avg_orders_per_customer,
    ROUND(AVG(total_spend), 2) AS avg_lifetime_spend,
    ROUND(AVG(avg_order_value), 2) AS overall_avg_order_value
FROM globalmart.gold.gold_customer_summary
GROUP BY is_active;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS globalmart.meta.watermark_metadata (
    table_name STRING,
    last_watermark TIMESTAMP,
    last_processed_at TIMESTAMP,
    records_processed BIGINT,
    status STRING
)
USING DELTA;

In [0]:
# ==============================================================================
# TASK 5.5: INCREMENTAL LOADER WITH WATERMARK TRACKING
# ==============================================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timedelta

def run_incremental_orders_load(
    source_table="globalmart.bronze.bronze_orders",
    target_table="globalmart.silver.orders_incremental",
    watermark_table="globalmart.meta.watermark_metadata",
    lookback_days=1
):
    print("=" * 80)
    print(f"STARTING INCREMENTAL LOAD: {source_table} -> {target_table}")
    print("=" * 80)

    # 1. Ensure Target Silver Table Exists
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {target_table} (
            order_id STRING,
            customer_id STRING,
            order_status STRING,
            order_purchase_timestamp TIMESTAMP,
            order_approved_at TIMESTAMP,
            order_delivered_carrier_date TIMESTAMP,
            order_delivered_customer_date TIMESTAMP,
            order_estimated_delivery_date TIMESTAMP,
            ingested_at TIMESTAMP
        )
        USING DELTA
    """)

    # 2. Fetch Last Successful Watermark Timestamp
    watermark_df = (
        spark.table(watermark_table)
        .filter((F.col("table_name") == target_table) & (F.col("status") == "SUCCESS"))
        .select(F.max("last_watermark").alias("max_watermark"))
    )

    result = watermark_df.collect()
    last_watermark = result[0]["max_watermark"] if result and result[0]["max_watermark"] else datetime(1970, 1, 1)

    # Calculate Effective Watermark with Lookback Window
    effective_watermark = last_watermark - timedelta(days=lookback_days) if last_watermark > datetime(1970, 1, 1) else last_watermark
    
    print(f"Previous Watermark  : {last_watermark}")
    print(f"Lookback Window     : {lookback_days} Day(s)")
    print(f"Effective Watermark : {effective_watermark}")

    # 3. Read Source Incremental Batch
    source_df = (
        spark.table(source_table)
        .filter(F.col("order_purchase_timestamp") > F.lit(effective_watermark))
        .withColumn("ingested_at", F.current_timestamp())
    )

    batch_count = source_df.count()
    print(f"Fetched Records     : {batch_count:,}")

    if batch_count == 0:
        print("No new records found to process.")
        return

    # Extract New High Watermark (Max timestamp in batch)
    next_watermark = source_df.agg(F.max("order_purchase_timestamp")).collect()[0][0]

    # 4. Perform Delta MERGE into Target Silver Table
    target_delta = DeltaTable.forName(spark, target_table)

    (
        target_delta.alias("target")
        .merge(
            source=source_df.alias("source"),
            condition="target.order_id = source.order_id"
        )
        .whenMatchedUpdate(
            condition="source.order_purchase_timestamp >= target.order_purchase_timestamp",
            set={
                "customer_id": "source.customer_id",
                "order_status": "source.order_status",
                "order_purchase_timestamp": "source.order_purchase_timestamp",
                "order_approved_at": "source.order_approved_at",
                "order_delivered_carrier_date": "source.order_delivered_carrier_date",
                "order_delivered_customer_date": "source.order_delivered_customer_date",
                "order_estimated_delivery_date": "source.order_estimated_delivery_date",
                "ingested_at": "source.ingested_at"
            }
        )
        .whenNotMatchedInsert(
            values={
                "order_id": "source.order_id",
                "customer_id": "source.customer_id",
                "order_status": "source.order_status",
                "order_purchase_timestamp": "source.order_purchase_timestamp",
                "order_approved_at": "source.order_approved_at",
                "order_delivered_carrier_date": "source.order_delivered_carrier_date",
                "order_delivered_customer_date": "source.order_delivered_customer_date",
                "order_estimated_delivery_date": "source.order_estimated_delivery_date",
                "ingested_at": "source.ingested_at"
            }
        )
        .execute()
    )

    # 5. Commit Entry into Watermark Metadata Table
    meta_entry = spark.createDataFrame([(
        target_table,
        next_watermark,
        datetime.now(),
        batch_count,
        "SUCCESS"
    )], schema="table_name STRING, last_watermark TIMESTAMP, last_processed_at TIMESTAMP, records_processed BIGINT, status STRING")

    meta_entry.write.mode("append").saveAsTable(watermark_table)

    print("-" * 80)
    print("INCREMENTAL LOAD COMPLETED SUCCESSFULLY")
    print(f"Updated Watermark   : {next_watermark}")
    print(f"Records Processed   : {batch_count:,}")
    print("=" * 80)

# Run Incremental Load Test
run_incremental_orders_load(lookback_days=1)

In [0]:
%sql
SELECT 
    table_name,
    last_watermark,
    last_processed_at,
    records_processed,
    status
FROM globalmart.meta.watermark_metadata
ORDER BY last_processed_at DESC;

In [0]:
%sql
SELECT 
    COUNT(*) AS total_incremental_orders,
    MIN(order_purchase_timestamp) AS min_order_date,
    MAX(order_purchase_timestamp) AS max_order_date,
    MAX(ingested_at) AS latest_batch_time
FROM globalmart.silver.orders_incremental;